# Experiment 4: Harmformer ($SE(2)$-Equivariant Encoder)

## Symmetry Group

The **special Euclidean group** $SE(2) = SO(2) \ltimes \mathbb{R}^2$ is the group of rigid motions of the plane. An element $(R_\alpha, t)$ rotates spatial coordinates by angle $\alpha$ and translates by $t \in \mathbb{R}^2$.

Since $D_4 \subset SE(2)$, this is the most general symmetry group in our framework.

## Architecture (Karella et al., 2024)

Achieves $SO(2)$-invariance via **orbit-mean pooling** (Reynolds operator). For $N$ uniformly spaced rotation angles $\{k \cdot 360^\circ / N\}_{k=0}^{N-1}$:

$$z = \varphi_\theta(x) = \frac{1}{N} \sum_{k=0}^{N-1} \text{backbone}\!\left(R_{k \cdot 360^\circ/N} \cdot x\right)$$

This is the **Reynolds operator** applied to the backbone output, which projects onto the $SO(2)$-invariant subspace.

**Invariance guarantee:**

$$\varphi_\theta(R_\alpha \cdot x) = \varphi_\theta(x) \quad \forall \alpha \in [0^\circ, 360^\circ)$$

(exact in the limit $N \to \infty$, approximate for finite $N$)

## Invariance Propagation

If $\varphi_\theta$ is $G$-invariant, then:

$$f_\theta(g \cdot x_1, g \cdot x_2) = f_\theta(x_1, x_2) \quad \forall g \in G$$

## Loss

$$\mathcal{L} = \mathcal{L}_{\mathrm{BCE}} + \lambda\,\mathcal{L}_{\mathrm{contr}}^{L^2}$$

**Isometry invariance of $\mathcal{L}_{\mathrm{contr}}^{L^2}$:** For orthogonal $U$, $\|Uz_1 - Uz_2\|_2 = \|z_1 - z_2\|_2$.

## Runs

| Run | Augmentation | $\lambda$ | $N_{\text{rots}}$ |
|-----|-------------|----------|------------------|
| `harmformer_no_reg` | Block P only | 0.0 | 8 |
| `harmformer_reg` | Block P only | 0.1 | 8 |

**Key hypothesis**: $SE(2)$-invariant encoder should achieve highest recall on arbitrary rotation angles (R90, R180, R270) while maintaining competitive FPR.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import yaml
import logging
import matplotlib.pyplot as plt

from src.utils.seed import set_seed
from src.data_module.augmentations import build_train_transform, build_val_transform
from src.data_module.coco_dataset import build_coco_dataloaders
from src.data_module.domainnet_dataset import DomainNetEvalDataset
from src.encoders import EncoderFactory
from src.losses.composite import CompositeLoss
from src.models.siamese import SiameseNet
from src.training.trainer import Trainer
from src.validation.evaluator import DomainNetEvaluator
from src.validation.tsne import compute_tsne_embeddings, plot_tsne
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

set_seed(cfg['seed'])
train_tfm = build_train_transform(cfg)
val_tfm = build_val_transform(cfg)
dataloaders = build_coco_dataloaders(cfg, train_tfm, val_tfm)

In [ ]:
histories = {}
models = {}

for run_name, lam in [('harmformer_no_reg', 0.0), ('harmformer_reg', 0.1)]:
    print(f'\n{"="*60}')
    print(f'Run: {run_name}, lambda={lam}')
    print(f'{"="*60}')
    
    set_seed(cfg['seed'])
    encoder = EncoderFactory('harmformer', freeze=True, n_rots=8)
    model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)
    
    criterion = CompositeLoss(
        bce_weight_pos=0.3, bce_weight_neg=0.7,
        contrastive_margin=1.0, lambda_reg=lam,
    )
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
    
    trainer = Trainer(
        model=model, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler, device=device,
        checkpoint_dir=f'../outputs/checkpoints/{run_name}',
    )
    histories[run_name] = trainer.fit(
        dataloaders['train'], dataloaders['val'],
        num_epochs=cfg['training']['num_epochs'],
        run_name=run_name,
    )
    models[run_name] = model

## DomainNet Evaluation

In [ ]:
domainnet_ds = DomainNetEvalDataset(
    root=cfg['data']['domainnet_root'],
    domains=cfg['data']['domainnet_domains'],
    pairs_per_domain=cfg['data']['pairs_per_domain'],
)

for run_name, model in models.items():
    print(f'\n--- {run_name} ---')
    evaluator = DomainNetEvaluator(model=model, dataset=domainnet_ds, device=device)
    report = evaluator.run()
    print(json.dumps(report, indent=2))

## t-SNE: Harmformer Embeddings

The Reynolds-averaged embeddings should show tighter clustering of geometric variants of the same image compared to non-equivariant encoders.

In [ ]:
last_model = list(models.values())[-1]
tsne_loader = DataLoader(domainnet_ds, batch_size=32, shuffle=False, num_workers=2)
coords, domains = compute_tsne_embeddings(last_model, tsne_loader, device=device)
fig = plot_tsne(coords, domains, title='t-SNE: Harmformer ($SE(2)$)',
                save_path='../outputs/figures/harmformer_tsne.pdf')
plt.show()

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    epochs = [h['epoch'] for h in hist]
    axes[0].plot(epochs, [h['train']['loss'] for h in hist], label=f'{name} (train)')
    axes[1].plot(epochs, [h['val']['loss'] for h in hist], label=f'{name} (val)')
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
fig.tight_layout()
fig.savefig('../outputs/figures/harmformer_curves.pdf', dpi=300)
plt.show()